# Compare MERRA-PRISM Inference With PRISM Observations

This notebook reads a MERRA-PRISM YAML configuration, finds the matching inference outputs, and compares them with daily PRISM files over the configured spatial subdomain.

Outputs include climatology comparison maps and GIF animations for `ppt`, `tmax`, and `tmin`.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import xarray as xr
import yaml

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_AVAILABLE = True
except ImportError:
    ccrs = None
    cfeature = None
    CARTOPY_AVAILABLE = False

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "granitewxc").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "examples" / "MERRA_PRISM"))
    sys.path.insert(0, str(REPO_ROOT))

from merra_prism_utils import case_output_dir, get_case_name, resolve_path

print(f"Repository root: {REPO_ROOT}")
print(f"Cartopy available: {CARTOPY_AVAILABLE}")

## User Parameters

In [ ]:
# ===================== USER PARAMETERS (EDIT ME) =====================
CONFIG_PATH = REPO_ROOT / "examples" / "MERRA_PRISM" / "MERRA_PRISM_subdomain.yaml"

# Leave as None to use data.target_dir and inference.output_dir from the YAML.
PRISM_ROOT = None
INFERENCE_OUTPUT_ROOT = None

# Climatology period. None means use the YAML inference period, clipped to files found.
# For faster first runs, set a smaller range such as "2016-01-01" to "2016-12-31".
CLIMO_START = None
CLIMO_END = None

# GIF period should stay modest because every frame reads PRISM + inference data.
GIF_START = "2025-07-01"
GIF_END = "2025-07-31"
GIF_FPS = 2

OUTPUT_DIR = REPO_ROOT / "examples" / "MERRA_PRISM" / "experiments" / "comparison_plots"
MAKE_GIFS = True

# Map backgrounds use Cartopy PlateCarree when available.
USE_CARTOPY = True
SHOW_COUNTIES = False  # County boundaries can trigger a Natural Earth download.
MAP_FEATURE_SCALE = "10m"  # "10m", "50m", or "110m"
COUNTY_FEATURE_SCALE = "10m"
# ====================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Comparison artifacts will be written to: {OUTPUT_DIR}")

## Load Configuration

In [ ]:
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    cfg = yaml.safe_load(fh)

data_cfg = cfg["data"]
case_name = get_case_name(cfg)
target_variables = list(data_cfg.get("target_variables", ["ppt", "tmax", "tmin"]))
spatial_subset = data_cfg.get("spatial_subset", {}) or {}

prism_root = Path(PRISM_ROOT) if PRISM_ROOT is not None else Path(data_cfg["target_dir"])
inference_root = (
    Path(INFERENCE_OUTPUT_ROOT)
    if INFERENCE_OUTPUT_ROOT is not None
    else resolve_path(cfg.get("inference", {}).get("output_dir", "./examples/MERRA_PRISM/experiments/inference_output"))
)
inference_dir = case_output_dir(inference_root, case_name)

yaml_start = pd.Timestamp(cfg["dates"]["inference"]["start"])
yaml_end = pd.Timestamp(cfg["dates"]["inference"]["end"])

print(f"Config: {CONFIG_PATH}")
print(f"Case name: {case_name}")
print(f"Variables: {target_variables}")
print(f"PRISM root: {prism_root}")
print(f"Inference dir: {inference_dir}")
print(f"YAML inference period: {yaml_start.date()} to {yaml_end.date()}")
print(f"Spatial subset: {spatial_subset}")

## Helper Functions

In [ ]:
DATE_RE = re.compile(r"_(\d{8})\.nc$")

VAR_LABELS = {
    "ppt": "Precipitation",
    "tmax": "Maximum temperature",
    "tmin": "Minimum temperature",
}
VAR_UNITS = {"ppt": "mm/day", "tmax": "deg C", "tmin": "deg C"}
VAR_CMAPS = {"ppt": "viridis", "tmax": "magma", "tmin": "plasma"}


def inference_file(date):
    token = pd.Timestamp(date).strftime("%Y%m%d")
    return inference_dir / f"{case_name}_inference_{token}.nc"


def prism_file(var, date):
    stamp = pd.Timestamp(date)
    token = stamp.strftime("%Y%m%d")
    return prism_root / var / str(stamp.year) / f"prism_{var}_us_30s_{token}.nc"


def dates_from_inference_files():
    dates = []
    for path in sorted(inference_dir.glob(f"{case_name}_inference_*.nc")):
        match = DATE_RE.search(path.name)
        if match:
            dates.append(pd.Timestamp(match.group(1)))
    return pd.DatetimeIndex(dates)


def subset_da(da):
    lat_min = spatial_subset.get("lat_min")
    lat_max = spatial_subset.get("lat_max")
    lon_min = spatial_subset.get("lon_min")
    lon_max = spatial_subset.get("lon_max")
    if spatial_subset.get("enabled", False) and None not in (lat_min, lat_max):
        lat0, lat1 = sorted((float(lat_min), float(lat_max)))
        da = da.sel(lat=slice(lat0, lat1)) if da.lat[0] < da.lat[-1] else da.sel(lat=slice(lat1, lat0))
    if spatial_subset.get("enabled", False) and None not in (lon_min, lon_max):
        lon0, lon1 = sorted((float(lon_min), float(lon_max)))
        da = da.sel(lon=slice(lon0, lon1)) if da.lon[0] < da.lon[-1] else da.sel(lon=slice(lon1, lon0))
    return da


def open_prism_day(var, date, target_lat=None, target_lon=None):
    path = prism_file(var, date)
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path) as ds:
        name = var if var in ds.data_vars else "Band1"
        da = ds[name].astype("float32")
        da = subset_da(da)
        if target_lat is not None and target_lon is not None:
            expected_lat = np.asarray(getattr(target_lat, "values", target_lat), dtype=np.float64)
            expected_lon = np.asarray(getattr(target_lon, "values", target_lon), dtype=np.float64)
            observed_lat = np.asarray(da.lat.values, dtype=np.float64)
            observed_lon = np.asarray(da.lon.values, dtype=np.float64)
            if not np.array_equal(observed_lat, expected_lat) or not np.array_equal(observed_lon, expected_lon):
                raise ValueError("Inference and PRISM coordinates are not the exact canonical grid; re-run preprocessing/inference")
        return da.load()


def open_inference_day(date):
    path = inference_file(date)
    if not path.exists():
        raise FileNotFoundError(path)
    ds = xr.open_dataset(path)
    # Inference is already on the case's canonical subset grid. Re-slicing
    # float coordinates can drop an endpoint, so validate it unchanged.
    return ds.load()


def date_range(start, end):
    return pd.date_range(pd.Timestamp(start), pd.Timestamp(end), freq="D")


def existing_common_dates(start, end):
    dates = []
    for date in date_range(start, end):
        if not inference_file(date).exists():
            continue
        if all(prism_file(var, date).exists() for var in target_variables):
            dates.append(date)
    return pd.DatetimeIndex(dates)


def symmetric_limits(da, q=0.98):
    vals = np.asarray(da.values).ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return -1.0, 1.0
    lim = float(np.nanquantile(np.abs(vals), q))
    if not np.isfinite(lim) or lim == 0:
        lim = 1.0
    return -lim, lim


def field_limits(*arrays, q=0.02):
    vals = np.concatenate([np.asarray(a.values).ravel() for a in arrays])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    lo = float(np.nanquantile(vals, q))
    hi = float(np.nanquantile(vals, 1 - q))
    if lo == hi:
        hi = lo + 1.0
    return lo, hi


def cartopy_enabled():
    return bool(USE_CARTOPY and CARTOPY_AVAILABLE)


def map_subplot_kw():
    return {"projection": ccrs.PlateCarree()} if cartopy_enabled() else {}


def map_plot_kwargs():
    return {"transform": ccrs.PlateCarree()} if cartopy_enabled() else {}


def map_extent(da, pad_fraction=0.02):
    lon0 = float(da.lon.min())
    lon1 = float(da.lon.max())
    lat0 = float(da.lat.min())
    lat1 = float(da.lat.max())
    lon_pad = max((lon1 - lon0) * pad_fraction, 0.01)
    lat_pad = max((lat1 - lat0) * pad_fraction, 0.01)
    return [lon0 - lon_pad, lon1 + lon_pad, lat0 - lat_pad, lat1 + lat_pad]


def add_map_background(ax, da):
    if not cartopy_enabled():
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        return

    data_crs = ccrs.PlateCarree()
    ax.set_extent(map_extent(da), crs=data_crs)
    ax.coastlines(resolution=MAP_FEATURE_SCALE, linewidth=0.7, color="0.15")
    ax.add_feature(cfeature.BORDERS.with_scale(MAP_FEATURE_SCALE), linewidth=0.5, edgecolor="0.25")
    ax.add_feature(cfeature.STATES.with_scale(MAP_FEATURE_SCALE), linewidth=0.55, edgecolor="0.2")
    if SHOW_COUNTIES:
        counties = cfeature.NaturalEarthFeature(
            "cultural",
            "admin_2_counties",
            COUNTY_FEATURE_SCALE,
            facecolor="none",
        )
        ax.add_feature(counties, linewidth=0.25, edgecolor="0.35")
    gl = ax.gridlines(draw_labels=True, linewidth=0.25, color="0.45", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}

## Discover Available Dates

In [ ]:
inference_dates = dates_from_inference_files()
if len(inference_dates) == 0:
    raise FileNotFoundError(f"No inference files found in {inference_dir}")

climo_start = pd.Timestamp(CLIMO_START) if CLIMO_START else max(yaml_start, inference_dates.min())
climo_end = pd.Timestamp(CLIMO_END) if CLIMO_END else min(yaml_end, inference_dates.max())
common_dates = existing_common_dates(climo_start, climo_end)

if len(common_dates) == 0:
    raise FileNotFoundError(f"No common inference/PRISM dates found between {climo_start.date()} and {climo_end.date()}")

print(f"Inference files: {len(inference_dates)} ({inference_dates.min().date()} to {inference_dates.max().date()})")
print(f"Climatology common dates: {len(common_dates)} ({common_dates.min().date()} to {common_dates.max().date()})")

sample = open_inference_day(common_dates[0])
print(sample)
sample.close()

## Compute Climatology Maps

This cell streams through one date at a time. It is slower than loading all data at once, but keeps memory use predictable for full multi-year comparisons.

In [ ]:
sums = {}
sumsq = {}
counts = {}

for i, date in enumerate(common_dates, start=1):
    if i == 1 or i % 100 == 0 or i == len(common_dates):
        print(f"[{i:>5}/{len(common_dates)}] {date.date()}")

    inf_ds = open_inference_day(date)
    target_lat = inf_ds.lat
    target_lon = inf_ds.lon

    try:
        for var in target_variables:
            inf = inf_ds[var].isel(time=0).astype("float32")
            obs = open_prism_day(var, date, target_lat=target_lat, target_lon=target_lon)
            diff = inf - obs

            if var not in sums:
                zeros = xr.zeros_like(inf, dtype="float64")
                sums[var] = {"inference": zeros.copy(), "prism": zeros.copy(), "bias": zeros.copy()}
                sumsq[var] = xr.zeros_like(inf, dtype="float64")
                counts[var] = xr.zeros_like(inf, dtype="int32")

            valid = np.isfinite(inf) & np.isfinite(obs)
            sums[var]["inference"] = sums[var]["inference"] + inf.where(valid, 0.0)
            sums[var]["prism"] = sums[var]["prism"] + obs.where(valid, 0.0)
            sums[var]["bias"] = sums[var]["bias"] + diff.where(valid, 0.0)
            sumsq[var] = sumsq[var] + (diff ** 2).where(valid, 0.0)
            counts[var] = counts[var] + valid.astype("int32")
    finally:
        inf_ds.close()

climo = {}
for var in target_variables:
    count = counts[var].where(counts[var] > 0)
    climo[var] = {
        "inference": (sums[var]["inference"] / count).astype("float32"),
        "prism": (sums[var]["prism"] / count).astype("float32"),
        "bias": (sums[var]["bias"] / count).astype("float32"),
        "rmse": np.sqrt(sumsq[var] / count).astype("float32"),
        "count": counts[var],
    }

print("Done.")

## Plot Climatology Maps

In [ ]:
def plot_climatology(var):
    inf = climo[var]["inference"]
    obs = climo[var]["prism"]
    bias = climo[var]["bias"]
    rmse = climo[var]["rmse"]

    vmin, vmax = field_limits(inf, obs)
    bmin, bmax = symmetric_limits(bias)
    rmax = float(np.nanquantile(rmse.values, 0.98))
    if not np.isfinite(rmax) or rmax == 0:
        rmax = 1.0

    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True, subplot_kw=map_subplot_kw())
    panels = [
        (axes[0, 0], inf, "Inference mean", VAR_CMAPS.get(var, "viridis"), vmin, vmax),
        (axes[0, 1], obs, "PRISM mean", VAR_CMAPS.get(var, "viridis"), vmin, vmax),
        (axes[1, 0], bias, "Inference - PRISM", "RdBu_r", bmin, bmax),
        (axes[1, 1], rmse, "RMSE", "inferno", 0.0, rmax),
    ]
    for ax, da, title, cmap, lo, hi in panels:
        im = da.plot(ax=ax, x="lon", y="lat", cmap=cmap, vmin=lo, vmax=hi, add_colorbar=False, **map_plot_kwargs())
        add_map_background(ax, da)
        ax.set_title(title)
        cbar = fig.colorbar(im, ax=ax, shrink=0.86)
        cbar.set_label(VAR_UNITS.get(var, ""))

    start = common_dates.min().strftime("%Y-%m-%d")
    end = common_dates.max().strftime("%Y-%m-%d")
    fig.suptitle(f"{VAR_LABELS.get(var, var)} climatology: {case_name} ({start} to {end})")
    out = OUTPUT_DIR / f"{case_name}_{var}_climatology_comparison.png"
    fig.savefig(out, dpi=160)
    print(out)
    return fig


for var in target_variables:
    plot_climatology(var)
plt.show()

## Summary Metrics

In [ ]:
rows = []
for var in target_variables:
    rows.append({
        "variable": var,
        "mean_inference": float(climo[var]["inference"].mean(skipna=True)),
        "mean_prism": float(climo[var]["prism"].mean(skipna=True)),
        "mean_bias": float(climo[var]["bias"].mean(skipna=True)),
        "spatial_mean_rmse": float(climo[var]["rmse"].mean(skipna=True)),
        "valid_grid_cells": int((climo[var]["count"] > 0).sum()),
    })

metrics = pd.DataFrame(rows)
metrics_path = OUTPUT_DIR / f"{case_name}_climatology_metrics.csv"
metrics.to_csv(metrics_path, index=False)
print(metrics_path)
metrics

## GIF Animations

Each GIF frame shows inference, PRISM, and their difference for a single day. Keep this period short for responsive rendering.

In [ ]:
def load_animation_frames(var, dates):
    frames = []
    for i, date in enumerate(dates, start=1):
        print(f"{var}: frame {i}/{len(dates)} {date.date()}")
        inf_ds = open_inference_day(date)
        try:
            inf = inf_ds[var].isel(time=0).astype("float32")
            obs = open_prism_day(var, date, target_lat=inf_ds.lat, target_lon=inf_ds.lon)
            inf = inf.where(np.isfinite(obs))
            frames.append((date, inf, obs, inf - obs))
        finally:
            inf_ds.close()
    return frames


def make_comparison_gif(var, dates):
    frames = load_animation_frames(var, dates)
    if not frames:
        print(f"No frames for {var}")
        return None

    vmin, vmax = field_limits(*[item for _, inf, obs, _ in frames for item in (inf, obs)])
    diffs = [diff for _, _, _, diff in frames]
    dmin, dmax = symmetric_limits(xr.concat(diffs, dim="frame"))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), constrained_layout=True, subplot_kw=map_subplot_kw())
    titles = ["Inference", "PRISM", "Inference - PRISM"]
    cmaps = [VAR_CMAPS.get(var, "viridis"), VAR_CMAPS.get(var, "viridis"), "RdBu_r"]
    limits = [(vmin, vmax), (vmin, vmax), (dmin, dmax)]

    images = []
    first_date, first_inf, first_obs, first_diff = frames[0]
    for ax, da, title, cmap, (lo, hi) in zip(axes, [first_inf, first_obs, first_diff], titles, cmaps, limits):
        im = da.plot(ax=ax, x="lon", y="lat", cmap=cmap, vmin=lo, vmax=hi, add_colorbar=False, **map_plot_kwargs())
        add_map_background(ax, da)
        ax.set_title(title)
        cbar = fig.colorbar(im, ax=ax, shrink=0.82)
        cbar.set_label(VAR_UNITS.get(var, ""))
        images.append(im)

    suptitle = fig.suptitle("")

    def update(frame_idx):
        date, inf, obs, diff = frames[frame_idx]
        for im, da in zip(images, [inf, obs, diff]):
            im.set_array(np.asarray(da.values).ravel())
        suptitle.set_text(f"{VAR_LABELS.get(var, var)} daily comparison: {date.strftime('%Y-%m-%d')}")
        return images + [suptitle]

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 / GIF_FPS, blit=False)
    out = OUTPUT_DIR / f"{case_name}_{var}_{dates.min().strftime('%Y%m%d')}_{dates.max().strftime('%Y%m%d')}.gif"
    anim.save(out, writer=PillowWriter(fps=GIF_FPS))
    plt.close(fig)
    print(out)
    return out


if MAKE_GIFS:
    gif_dates = existing_common_dates(GIF_START, GIF_END)
    if len(gif_dates) == 0:
        raise FileNotFoundError(f"No common inference/PRISM dates found for GIF period {GIF_START} to {GIF_END}")
    print(f"GIF dates: {len(gif_dates)} ({gif_dates.min().date()} to {gif_dates.max().date()})")
    gif_paths = [make_comparison_gif(var, gif_dates) for var in target_variables]
else:
    gif_paths = []

gif_paths